In [1]:
# ============================================================
# KAFKA ARCHITECTURE — DEEP DIVE (Jupyter Notebook)
# From Basic to Advanced — Cell by Cell
# ============================================================
# CELL 1: Install & Verify Connection to Kafka Broker
# ============================================================

# Step 1: Install library (run once)
import subprocess
subprocess.run(["pip", "install", "kafka-python"], capture_output=True)

# Step 2: Import & test connection
from kafka import KafkaAdminClient

# KafkaAdminClient connects to the Broker
# 'localhost:9092' is your Broker address (the Kafka server running on your Mac)
admin = KafkaAdminClient(bootstrap_servers="localhost:9092")

# List all existing topics on the Broker
topics = admin.list_topics()

print("✅ Connected to Kafka Broker at localhost:9092")
print(f"📂 Topics found on Broker: {topics}")

admin.close()

✅ Connected to Kafka Broker at localhost:9092
📂 Topics found on Broker: ['demo-topic', '__consumer_offsets']


In [2]:
# ============================================================
# CELL 2: Topics & Partitions — The Logical Storage
# ============================================================
# Topic   = ek category (jaise database table)
# Partition = Topic ke tukde (speed aur parallelism ke liye)
# ============================================================

from kafka import KafkaAdminClient
from kafka.admin import NewTopic

admin = KafkaAdminClient(bootstrap_servers="localhost:9092")

# ── Create 3 Topics (real ML pipeline topics) ──────────────
topics_to_create = [
    NewTopic(name="user-clicks",       num_partitions=3, replication_factor=1),
    NewTopic(name="sensor-telemetry",  num_partitions=5, replication_factor=1),
    NewTopic(name="ml-predictions",    num_partitions=2, replication_factor=1),
]

# Create only if they don't already exist
existing = admin.list_topics()
new_ones = [t for t in topics_to_create if t.name not in existing]

if new_ones:
    admin.create_topics(new_topics=new_ones)
    print(f"✅ Created {len(new_ones)} new topics")
else:
    print("ℹ️  Topics already exist — skipping creation")

# ── Inspect Partitions of each topic ───────────────────────
from kafka import KafkaConsumer

consumer = KafkaConsumer(bootstrap_servers="localhost:9092")

print("\n📂 TOPIC → PARTITION MAP")
print("=" * 45)

all_topics = ["user-clicks", "sensor-telemetry", "ml-predictions"]

for topic in all_topics:
    partitions = consumer.partitions_for_topic(topic)
    print(f"  Topic: '{topic}'")
    print(f"  Partitions: {sorted(partitions)}")
    print(f"  Total: {len(partitions)} partitions")
    print("-" * 45)

consumer.close()
admin.close()

# ── Explanation ─────────────────────────────────────────────
print("""
💡 WHY PARTITIONS MATTER (ML Context):
   sensor-telemetry has 5 partitions
   → 5 sensors can write SIMULTANEOUSLY
   → 5 ML models can read SIMULTANEOUSLY
   → 5x throughput vs single partition!

   Rule of thumb:
   partitions = max(producers, consumers)
   you expect to run in parallel
""")

✅ Created 3 new topics

📂 TOPIC → PARTITION MAP
  Topic: 'user-clicks'
  Partitions: [0, 1, 2]
  Total: 3 partitions
---------------------------------------------
  Topic: 'sensor-telemetry'
  Partitions: [0, 1, 2, 3, 4]
  Total: 5 partitions
---------------------------------------------
  Topic: 'ml-predictions'
  Partitions: [0, 1]
  Total: 2 partitions
---------------------------------------------

💡 WHY PARTITIONS MATTER (ML Context):
   sensor-telemetry has 5 partitions
   → 5 sensors can write SIMULTANEOUSLY
   → 5 ML models can read SIMULTANEOUSLY
   → 5x throughput vs single partition!

   Rule of thumb:
   partitions = max(producers, consumers)
   you expect to run in parallel



In [3]:
# ============================================================
# CELL 3: Producer — Kafka mein message bhejne wala
# ============================================================
# Producer = Writer. Topic mein data push karta hai.
# ============================================================

from kafka import KafkaProducer
import json

# Create a Producer connected to our Broker
producer = KafkaProducer(
    bootstrap_servers="localhost:9092",
    value_serializer=lambda v: json.dumps(v).encode("utf-8")
    # value_serializer: Python dict → JSON → bytes
    # Kafka sirf bytes store karta hai, strings nahi
)

# Send 3 messages to 'sensor-telemetry' topic
messages = [
    {"sensor_id": "S1", "temp": 22.5},
    {"sensor_id": "S2", "temp": 35.1},
    {"sensor_id": "S3", "temp": 19.8},
]

for msg in messages:
    producer.send("sensor-telemetry", value=msg)
    print(f"📤 Sent: {msg}")

producer.flush()  # Wait until all messages are delivered
print("\n✅ All messages delivered to Broker!")

📤 Sent: {'sensor_id': 'S1', 'temp': 22.5}
📤 Sent: {'sensor_id': 'S2', 'temp': 35.1}
📤 Sent: {'sensor_id': 'S3', 'temp': 19.8}

✅ All messages delivered to Broker!


In [4]:
# ============================================================
# CELL 4: Consumer — Kafka se message padhne wala
# ============================================================
# Consumer = Reader. Topic se data pull karta hai.
# auto_offset_reset='earliest' → pehle message se padho
# consumer_timeout_ms=3000    → 3 sec baad automatically stop
# ============================================================

from kafka import KafkaConsumer
import json

consumer = KafkaConsumer(
    "sensor-telemetry",
    bootstrap_servers="localhost:9092",
    auto_offset_reset="earliest",       # beginning se padho
    group_id="learning-group",          # consumer group name
    value_deserializer=lambda m: json.loads(m.decode("utf-8")),
    consumer_timeout_ms=3000            # 3 sec baad stop
)

print("📥 Reading messages from 'sensor-telemetry'...\n")

for msg in consumer:
    print(f"  Partition : {msg.partition}")
    print(f"  Offset    : {msg.offset}")
    print(f"  Value     : {msg.value}")
    print("-" * 35)

consumer.close()
print("✅ Done reading!")

📥 Reading messages from 'sensor-telemetry'...

✅ Done reading!


In [6]:
# ============================================================
# CELL 5: Keys — Related messages ek hi partition mein bhejo
# ============================================================
# Key = message ka identity tag
# Same key → always same partition → guaranteed order!
# ML Use: ek user ke saare events ek hi partition mein
# ============================================================

from kafka import KafkaProducer
import json

producer = KafkaProducer(
    bootstrap_servers="localhost:9092",
    key_serializer=lambda k: k.encode("utf-8"),
    value_serializer=lambda v: json.dumps(v).encode("utf-8")
)

# Same user ka events — same partition mein jaane chahiye
events = [
    ("user_1", {"action": "login",    "page": "home"}),
    ("user_2", {"action": "purchase", "item": "shoes"}),
    ("user_1", {"action": "click",    "page": "about"}),  # user_1 again
    ("user_1", {"action": "logout",   "page": "home"}),   # user_1 again
]

for key, value in events:
    future = producer.send("user-clicks", key=key, value=value)
    record = future.get(timeout=10)  # wait for confirmation
    print(f"  Key={key} → Partition {record.partition}, Offset {record.offset}")

producer.flush()
print("\n✅ Notice: user_1 always lands on same partition!")

  Key=user_1 → Partition 0, Offset 0
  Key=user_2 → Partition 1, Offset 0
  Key=user_1 → Partition 0, Offset 1
  Key=user_1 → Partition 0, Offset 2

✅ Notice: user_1 always lands on same partition!


In [7]:
# ============================================================
# CELL 6: Offsets — Har message ka unique address
# ============================================================
# Offset = message ki position in partition (0, 1, 2, 3...)
# Consumer apna offset commit karta hai = "main yahan tak padh chuka"
# Crash ke baad wahi se resume kar sakta hai!
# ============================================================

from kafka import KafkaConsumer, TopicPartition
import json

consumer = KafkaConsumer(
    bootstrap_servers="localhost:9092",
    value_deserializer=lambda m: json.loads(m.decode("utf-8")),
    group_id="offset-demo-group",
    auto_offset_reset="earliest",
    enable_auto_commit=False  # manual control for learning
)

# Check beginning & end offset of each partition
tp0 = TopicPartition("user-clicks", 0)
tp1 = TopicPartition("user-clicks", 1)
tp2 = TopicPartition("user-clicks", 2)

consumer.assign([tp0, tp1, tp2])

# beginning offsets
begin = consumer.beginning_offsets([tp0, tp1, tp2])
end   = consumer.end_offsets([tp0, tp1, tp2])

print("📍 OFFSET MAP for 'user-clicks'\n")
for tp in [tp0, tp1, tp2]:
    print(f"  Partition {tp.partition} → start={begin[tp]}  end={end[tp]}  total={(end[tp]-begin[tp])} msgs")

# Manually seek to beginning of partition 2 and read
print("\n📖 Reading Partition 2 from offset 0...\n")
consumer.seek(tp2, 0)

for _ in range(end[tp2] - begin[tp2]):
    msg = next(consumer)
    print(f"  Offset {msg.offset} → {msg.value}")
    consumer.commit()  # bookmark this position manually
    print(f"  ✅ Committed offset {msg.offset + 1}")

consumer.close()

📍 OFFSET MAP for 'user-clicks'

  Partition 0 → start=0  end=3  total=3 msgs
  Partition 1 → start=0  end=1  total=1 msgs
  Partition 2 → start=0  end=0  total=0 msgs

📖 Reading Partition 2 from offset 0...



In [ ]:
# ============================================================
# CELL 7: Consumer Groups — Team mein kaam karo!
# ============================================================
# Same group_id = ek team. Kafka partitions baant deta hai.
# Partition 0 → Consumer A
# Partition 1 → Consumer B  
# Partition 2 → Consumer C
# Teeno parallel mein kaam karenge = 3x speed!
# ============================================================

from kafka import KafkaConsumer
from threading import Thread
import json

def consume(consumer_id, group_id):
    consumer = KafkaConsumer(
        "sensor-telemetry",
        bootstrap_servers="localhost:9092",
        group_id=group_id,                 # same group = same team
        auto_offset_reset="earliest",
        value_deserializer=lambda m: json.loads(m.decode("utf-8")),
        consumer_timeout_ms=3000
    )

    print(f"\n🟢 Consumer-{consumer_id} started (Group: {group_id})")

    for msg in consumer:
        print(f"  Consumer-{consumer_id} | Partition {msg.partition} | {msg.value}")

    print(f"🔴 Consumer-{consumer_id} done")
    consumer.close()

# Launch 3 consumers in same group — parallel threads
threads = []
for i in range(1, 4):
    t = Thread(target=consume, args=(i, "ml-team"))
    threads.append(t)
    t.start()

for t in threads:
    t.join()

print("\n✅ All consumers finished!")
print("💡 Notice: each partition handled by only ONE consumer")

In [8]:
# ============================================================
# CELL 8: Rebalancing — Kafka ka automatic recovery system
# ============================================================
# Jab ek consumer crash hota hai, Kafka uska partition
# automatically doosre consumer ko de deta hai.
# Yahi Kafka ko fault-tolerant banata hai!
# ============================================================

from kafka import KafkaConsumer
from threading import Thread
import json, time

results = []  # shared log

def consume(consumer_id, stop_after=None):
    consumer = KafkaConsumer(
        "sensor-telemetry",
        bootstrap_servers="localhost:9092",
        group_id="rebalance-demo",        # same group
        auto_offset_reset="earliest",
        value_deserializer=lambda m: json.loads(m.decode("utf-8")),
        consumer_timeout_ms=6000,
        session_timeout_ms=6000,          # detect failure in 6s
        heartbeat_interval_ms=2000
    )

    count = 0
    for msg in consumer:
        results.append(f"C{consumer_id} | Partition {msg.partition} | {msg.value['sensor_id']}")
        count += 1

        # Simulate Consumer-1 crashing after 1 message
        if stop_after and count >= stop_after:
            print(f"💥 Consumer-{consumer_id} CRASHED! (simulated)")
            break

    consumer.close()

# Step 1: Start 2 consumers
print("🚀 Step 1: Starting 2 consumers in same group...\n")
t1 = Thread(target=consume, args=(1, 1))  # crashes after 1 msg
t2 = Thread(target=consume, args=(2,))

t1.start()
time.sleep(1)
t2.start()

t1.join()
t2.join()

# Print results
print("\n📋 MESSAGE PROCESSING LOG:")
print("=" * 45)
for r in results:
    print(f"  {r}")

print("""
📊 WHAT HAPPENED:
  1. C1 and C2 started → Kafka split partitions
  2. C1 crashed        → Kafka detected via heartbeat
  3. Rebalancing       → C2 got C1's partitions too
  4. No message lost   → C2 continued from last offset

✅ This is Kafka's fault tolerance in action!
""")

🚀 Step 1: Starting 2 consumers in same group...


📋 MESSAGE PROCESSING LOG:
  C2 | Partition 0 | S3
  C2 | Partition 1 | S1
  C2 | Partition 4 | S2

📊 WHAT HAPPENED:
  1. C1 and C2 started → Kafka split partitions
  2. C1 crashed        → Kafka detected via heartbeat
  3. Rebalancing       → C2 got C1's partitions too
  4. No message lost   → C2 continued from last offset

✅ This is Kafka's fault tolerance in action!

